# Vista — treino no Colab

Treina as duas redes do Vista com dados do **Vista-Sim** e mede o resultado **dirigindo no simulador**.

| Rede | Entrada | Saída | Base pré-treinada |
|---|---|---|---|
| **Faixa** | imagem + manobra (frente/esquerda/direita) | `e_y`, `e_ψ`, curvatura à frente, distância ao cruzamento, "dentro do cruzamento?" | ResNet18 (ImageNet) |
| **Detector** | imagem | 6 placas da Figura 8 + semáforo vermelho/amarelo/verde | YOLO11n (COCO) |

A decisão no cruzamento **não** é aprendida: sai de `vista_sim/rules.py` (placa → manobra), a mesma regra que vai para a pista.

**Como usar:** `Ambiente de execução → Alterar o tipo → GPU` (L4 ou A100) e `Executar tudo`. Os modelos vão para o seu Google Drive.

**"Treinar exaustivamente" do jeito certo:** o que melhora a rede na pista real é **variedade** (mais episódios = mais rotas, luz, câmera, carro sorteados), não repetir as mesmas imagens por muitas épocas — isso só decora a cara do simulador. Para ir mais longe, aumente `EPISODIOS`, não `EPOCAS_FAIXA`.

> **Economizar unidades de computação:** gerar os dados usa só CPU. Dá para gerar num ambiente **sem GPU** (`FONTE_DADOS = "gerar"` + `SALVAR_DATASET_NO_DRIVE = True`), e depois treinar num ambiente com GPU usando `FONTE_DADOS = "drive"`.

## 1. Parâmetros

In [ ]:
REPO = "https://github.com/Gustavo-dev04/C.A.FATEC.R.B.git"
BRANCH = "claude/autonomous-car-repo-data-lzwn7t"

# --- dados -----------------------------------------------------------------
FONTE_DADOS = "gerar"            # "gerar" aqui | "drive" (usa dataset.zip salvo antes)
SALVAR_DATASET_NO_DRIVE = False  # ~4 GB com 150 episódios
EPISODIOS = 150                  # cada um: uma rota sorteada inteira (~1.000 imagens)
SEMENTE_INICIAL = 0
PASSO = 4                        # salva 1 quadro a cada 4 (30 Hz → 7,5 Hz)
VISTAS_EXTRAS = 1                # vistas deslocadas (carro fora do centro) por quadro

# --- rede de faixa ---------------------------------------------------------
EPOCAS_FAIXA = 20
LOTE_FAIXA = 256
TAMANHO = (240, 320)             # (altura, largura) da entrada da rede
LR_FAIXA = 5e-4

# --- detector --------------------------------------------------------------
TREINAR_DETECTOR = True
MODELO_YOLO = "yolo11n.pt"       # pré-treinado no COCO; "n" = nano, cabe na Orin Nano
EPOCAS_YOLO = 80
IMG_YOLO = 416                   # a câmera processa 410×308 → múltiplo de 32

# --- avaliação no simulador (sementes NUNCA vistas no treino) --------------
SEMENTES_AVALIACAO = list(range(100000, 100010))

PASTA_DRIVE = "/content/drive/MyDrive/vista"

## 2. Ambiente

Clona o repositório e instala o que falta. O Colab já traz PyTorch, OpenCV, NumPy e PyYAML.

Se o repositório for **privado**: crie um token no GitHub (Settings → Developer settings → Personal access tokens, só leitura do repositório) e guarde nos **Segredos** do Colab (ícone de chave 🔑 à esquerda) com o nome `GITHUB_TOKEN`. O token nunca aparece no notebook.

In [ ]:
import os, subprocess, sys, json, time
from pathlib import Path

REPO_DIR = Path("/content/vista")
if not REPO_DIR.exists():
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("Clone público falhou — tentando com o GITHUB_TOKEN dos Segredos do Colab...")
        from google.colab import userdata
        url = REPO.replace("https://", f"https://x-access-token:{userdata.get('GITHUB_TOKEN')}@")
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO_DIR)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("não consegui clonar o repositório (token sem acesso?)")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT)

In [ ]:
%pip install -q onnx onnxruntime ultralytics
import torch, cv2, numpy as np
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sem GPU")
print("CPUs:", os.cpu_count())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAIDA = Path(PASTA_DRIVE) / "modelos" / time.strftime("%Y-%m-%d_%H%M")
SAIDA.mkdir(parents=True, exist_ok=True)
print("tudo que importa vai para:", SAIDA)

## 3. Dados

O oráculo dirige `EPISODIOS` rotas sorteadas (cidade, placas, luz, carro, IMU e montagem da câmera) e o simulador grava. A mesma semente + o mesmo commit geram **exatamente** o mesmo dataset — não precisa guardar, dá para regerar.

Tempo: ~30 s por episódio por núcleo de CPU.

In [ ]:
DADOS = Path("/content/dataset")
ZIP_DRIVE = Path(PASTA_DRIVE) / "dataset.zip"

if (DADOS / "meta.json").exists():
    print("dataset já existe em", DADOS)
elif FONTE_DADOS == "drive":
    print("copiando", ZIP_DRIVE, "...")
    !unzip -q "{ZIP_DRIVE}" -d /content
else:
    est = EPISODIOS * 30 / max(os.cpu_count(), 1) / 60
    print(f"gerando {EPISODIOS} episódios em {os.cpu_count()} núcleos (~{est:.0f} min)...")
    !python -m tools.gerar_dataset --out {DADOS} --episodios {EPISODIOS} --semente-inicial {SEMENTE_INICIAL} --passo {PASSO} --vistas-extras {VISTAS_EXTRAS}

meta = json.loads((DADOS / "meta.json").read_text())
print(f"{meta['episodios']} episódios, {meta['imagens']} imagens, gerado no commit {meta['commit']}")
if FONTE_DADOS == "gerar" and SALVAR_DATASET_NO_DRIVE and not ZIP_DRIVE.exists():
    !cd /content && zip -q -r -0 "{ZIP_DRIVE}" dataset
    print("salvo em", ZIP_DRIVE)

In [ ]:
# Amostras com os rótulos: confira antes de gastar GPU.
import csv, random
import matplotlib.pyplot as plt

linhas = list(csv.DictReader(open(DADOS / "rotulos.csv")))
objs = {json.loads(l)["arquivo"]: json.loads(l)["objetos"] for l in open(DADOS / "objetos.jsonl")}
random.seed(1)
fig, axs = plt.subplots(3, 4, figsize=(18, 10))
for ax, r in zip(axs.flat, random.sample(linhas, 12)):
    img = cv2.cvtColor(cv2.imread(str(DADOS / r["arquivo"])), cv2.COLOR_BGR2RGB)
    for o in objs.get(r["arquivo"], []):
        x0, y0, x1, y1 = o["bbox"]
        cv2.rectangle(img, (x0, y0), (x1, y1), (255, 255, 0), 1)
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{r['vista']} | e_y {float(r['e_y'])*100:+.1f} cm  e_ψ {np.degrees(float(r['e_psi'])):+.0f}°\n"
                 f"{r['manobra']} em {float(r['dist_cruzamento']):.1f} m  dentro={r['no_cruzamento']}", fontsize=9)
plt.tight_layout(); plt.show()

ey = np.array([float(r["e_y"]) for r in linhas])
print(f"e_y: p5 {np.percentile(ey, 5)*100:.1f} cm, p95 {np.percentile(ey, 95)*100:.1f} cm | "
      f"dentro do cruzamento: {np.mean([r['no_cruzamento'] == '1' for r in linhas])*100:.0f}% das imagens")

## 4. Rede de faixa

ResNet18 pré-treinada + a manobra (one-hot). Divisão treino/validação **por episódio** — quadros do mesmo episódio são quase iguais, misturar mediria memória.

Aumento de dados só de cor, desfoque, ruído e borrões: **nada de espelhar** (troca mão de direção e o sinal de `e_y`) nem girar (muda `e_ψ`).

In [ ]:
from treino.faixa import treinar, carregar, exportar_onnx, PreditorTorch

res = treinar(DADOS, SAIDA, epocas=EPOCAS_FAIXA, lote=LOTE_FAIXA, lr=LR_FAIXA, tamanho=TAMANHO,
              workers=min(os.cpu_count(), 12))

In [ ]:
h = res["historico"]
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
axs[0].plot([m["perda_treino"] for m in h], label="treino"); axs[0].plot([m["perda"] for m in h], label="validação")
axs[0].set_title("perda"); axs[0].legend()
axs[1].plot([m.get("e_y_cm_faixa") for m in h], label="na faixa"); axs[1].plot([m.get("e_y_cm_cruzamento") for m in h], label="no cruzamento")
axs[1].set_title("erro médio de e_y (cm)"); axs[1].legend()
axs[2].plot([m.get("e_psi_graus_faixa") for m in h], label="e_ψ na faixa (°)"); axs[2].plot([m["dist_cm"] / 10 for m in h], label="dist (dm)")
axs[2].set_title("e_ψ e distância"); axs[2].legend()
plt.show()
melhor = min(h, key=lambda m: m["perda"])
print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in melhor.items()}, indent=1))

In [ ]:
modelo, tam = carregar(res["melhor"], device="cuda" if torch.cuda.is_available() else "cpu")
info = exportar_onnx(modelo, tam, SAIDA / "faixa.onnx")
print(info)   # dif_max_torch_onnx deve ser ~1e-5 ou menos
modelo, tam = carregar(res["melhor"], device="cuda" if torch.cuda.is_available() else "cpu")

## 5. Detector de placas e semáforo (YOLO11n)

Do mesmo dataset: cada placa/semáforo visível vira uma caixa. Placa grande mas meio escondida tira a imagem do treino (sem rótulo ela ensinaria a **não** ver a placa).

⚠️ `fliplr=0`: espelhar a imagem transforma "virar à esquerda" em "virar à direita".

In [ ]:
if TREINAR_DETECTOR:
    from treino.deteccao import montar_yolo
    from ultralytics import YOLO

    YOLO_DIR = Path("/content/yolo")
    resumo = montar_yolo(DADOS, YOLO_DIR, val_frac=0.1)
    print(json.dumps({k: v for k, v in resumo.items() if k != "data_yaml"}, indent=1, ensure_ascii=False))

    det = YOLO(MODELO_YOLO)
    det.train(data=resumo["data_yaml"], epochs=EPOCAS_YOLO, imgsz=IMG_YOLO, batch=64, patience=20,
              fliplr=0.0, degrees=0.0, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, mosaic=1.0,
              workers=min(os.cpu_count(), 12), project="/content/yolo_runs", name="placas", exist_ok=True)

In [ ]:
if TREINAR_DETECTOR:
    best = Path("/content/yolo_runs/placas/weights/best.pt")
    det = YOLO(str(best))
    m = det.val(data=resumo["data_yaml"], imgsz=IMG_YOLO, split="val")
    print(f"mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f}")
    for i, nome in det.names.items():
        if i < len(m.box.maps):
            print(f"  {nome:28s} mAP50-95 {m.box.maps[i]:.3f}")
    onnx_det = det.export(format="onnx", imgsz=IMG_YOLO, opset=17, simplify=True)
    import shutil
    shutil.copy(best, SAIDA / "placas.pt")
    shutil.copy(onnx_det, SAIDA / "placas.onnx")
    print("detector salvo em", SAIDA)

## 6. A rede dirigindo no simulador

A métrica que importa: rotas **completas**. A rede de faixa fecha a malha (imagem → rede → `PilotoFaixa` → rpm de cada roda). Nesta etapa semáforo e manobra vêm da verdade — isola a rede de faixa.

Compare com o **teto**: o mesmo piloto com a verdade no lugar da rede.

In [ ]:
from treino.avaliacao import rodar

print("— teto (verdade de campo) —")
teto = rodar(None, SEMENTES_AVALIACAO)
print("\n— rede de faixa —")
aval = rodar(PreditorTorch(modelo, tam), SEMENTES_AVALIACAO)
for nome, r in (("teto", teto), ("rede", aval)):
    print(f"{nome}: {r['completas']}/{r['rotas']} rotas completas, progresso médio {r['progresso_medio']*100:.0f}%, "
          f"e_y RMS mediano {r['e_y_rms_cm_mediana']:.2f} cm, motivos {r['motivos']}")
(SAIDA / "avaliacao_simulador.json").write_text(json.dumps({"teto": teto, "rede": aval}, indent=2, ensure_ascii=False))

## 7. Salvar e levar para a Jetson

In [ ]:
import shutil
(SAIDA / "dataset_meta.json").write_text(json.dumps({**meta, "sementes": None}, indent=2, ensure_ascii=False))
(SAIDA / "LEIA-ME.txt").write_text(f"""Vista — modelos treinados no commit {COMMIT}
dataset: {meta['episodios']} episódios, {meta['imagens']} imagens (commit {meta['commit']})

faixa.onnx  + faixa.json   rede de faixa (contrato de entrada/saída no .json)
faixa_melhor.pt            checkpoint PyTorch (para continuar o treino)
placas.onnx + placas.pt    detector YOLO11n ({IMG_YOLO}×{IMG_YOLO})
avaliacao_simulador.json   a rede dirigindo no simulador

Na Jetson (o motor TensorRT é específico do aparelho — gere LÁ):
  /usr/src/tensorrt/bin/trtexec --onnx=faixa.onnx --fp16 --saveEngine=faixa.engine \\
      --minShapes=imagem:1x3x{TAMANHO[0]}x{TAMANHO[1]},manobra:1x3 \\
      --optShapes=imagem:1x3x{TAMANHO[0]}x{TAMANHO[1]},manobra:1x3 \\
      --maxShapes=imagem:1x3x{TAMANHO[0]}x{TAMANHO[1]},manobra:1x3
  /usr/src/tensorrt/bin/trtexec --onnx=placas.onnx --fp16 --saveEngine=placas.engine
""")
for f in sorted(SAIDA.iterdir()):
    print(f"{f.stat().st_size / 1e6:8.1f} MB  {f.name}")

## Próximos passos

1. **Ver a rede errar**: rotas incompletas em `avaliacao_simulador.json` → gerar mais dados parecidos (mais episódios, mais `VISTAS_EXTRAS`).
2. **Dados reais**: quando o carro estiver pronto, gravar na pista caseira e fazer *fine-tuning* a partir de `faixa_melhor.pt` com uma parte de imagens reais — o simulador nunca vai ser idêntico à câmera de verdade.
3. **Detector no laço**: trocar a manobra da verdade pela placa detectada + `rules.decide`.
4. **Nunca treinar na Jetson** — só rodar o TensorRT lá.